# Lab 4 — Econometric models (ARIMA, SARIMA, SARIMAX)

**Goal:** fit **SARIMA** \((p,d,q)(P,D,Q,m)\) and **SARIMAX** (with exogenous regressors) using `statsmodels`, evaluate on a held-out tail, and inspect **residuals** (ACF, Ljung–Box).

**Prerequisite:** `generate_data.ipynb` (includes `temp`, `promo` for ARIMAX).

Theory: `time_series_forecasting.md` (stationarity, differencing, seasonal differencing, white-noise residuals, ARIMA structure).

## 1. SARIMA in words

**AR** — \(y_t\) depends on past **levels** (or errors). **I** — **differencing** to stationarise mean. **MA** — depends on past **forecast errors**.

**Seasonal** block \((P,D,Q,m)\) — same idea at lag \(m\) (e.g. \(m=7\) days).

Orders \((p,d,q,P,D,Q)\) are usually chosen by **information criteria** (AIC/AICc/BIC) and residual diagnostics — not by p-hacking on the test set.

## 2. SARIMAX

When known **future** regressors exist (price, promo, temperature forecasts), include them:

\[
y_t = X_t'\beta + \text{ARIMA/SARIMA}(\varepsilon_t)
\]

Exogenous variables must be available for each forecast step (or scenario-based if uncertain).

## 3. Residual checks

Ideally residuals look like **white noise**: no structure in ACF, Ljung–Box not significant. If not, increase model order or add covariates — but watch **overfitting**.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

DATA_PATH = Path("data/synthetic_series.csv")
META_PATH = Path("data/synthetic_series_meta.json")
df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
df = df.set_index("date").asfreq("D")
m = int(json.load(open(META_PATH, encoding="utf-8")).get("seasonal_period", 7)) if META_PATH.exists() else 7

y = df["y"].astype(float)
exog = df[["temp", "promo"]].astype(float) if {"temp", "promo"}.issubset(df.columns) else None

TEST_H = 120
train, test = y.iloc[:-TEST_H], y.iloc[-TEST_H:]
print("Train", len(train), "Test", len(test), "m =", m)

In [ ]:
def rmse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))


def mae(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean(np.abs(a - b)))


# --- SARIMA (no exog) — moderate orders; adjust if convergence fails ---
order = (1, 1, 1)
sorder = (1, 0, 1, m)
mod_s = SARIMAX(train, order=order, seasonal_order=sorder, enforce_stationarity=True, enforce_invertibility=True)
res_s = mod_s.fit(disp=False, maxiter=200)
print("SARIMA AIC:", res_s.aic)
fc_s = res_s.get_forecast(steps=len(test))
pred_s = fc_s.predicted_mean
print("SARIMA test RMSE:", rmse(test.values, pred_s.values), "MAE:", mae(test.values, pred_s.values))

In [ ]:
if exog is not None:
    ex_tr, ex_te = exog.loc[train.index], exog.loc[test.index]
    mod_x = SARIMAX(train, exog=ex_tr, order=order, seasonal_order=sorder)
    res_x = mod_x.fit(disp=False, maxiter=200)
    print("SARIMAX AIC:", res_x.aic)
    pred_x = res_x.get_forecast(steps=len(test), exog=ex_te).predicted_mean
    print("SARIMAX test RMSE:", rmse(test.values, pred_x.values), "MAE:", mae(test.values, pred_x.values))
else:
    res_x = None
    print("No exogenous columns — skip SARIMAX.")

In [ ]:
# Residuals on training fit (in-sample)
resid = res_s.resid.dropna()
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(resid.index, resid.values, lw=0.6)
ax.set_title("SARIMA residuals (train)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 3))
plot_acf(resid, lags=min(40, len(resid) // 4), ax=ax)
ax.set_title("ACF of residuals")
plt.tight_layout()
plt.show()

lb = acorr_ljungbox(resid, lags=[10], return_df=True)
print(lb)

fig, ax = plt.subplots(figsize=(4, 4))
stats.probplot(resid, dist="norm", plot=ax)
ax.set_title("Normal Q-Q of residuals")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test.index, test.values, "k", label="Actual", lw=1)
ax.plot(test.index, pred_s.values, lw=0.9, label="SARIMA forecast")
if exog is not None:
    ax.plot(test.index, pred_x.values, lw=0.9, label="SARIMAX forecast")
ax.legend()
ax.set_title("Test period: multi-step forecasts from single origin")
plt.tight_layout()
plt.show()

## Note on evaluation

The block above uses **multi-step** forecasts from one fit at the train end — errors typically **grow** with horizon. For **rolling one-step** evaluation (like Lab 3), refit or update the state each day; that is a different experiment.

**Interview tip:** state the evaluation protocol clearly; compare models under the **same** protocol.

## Interview checklist

1. **Unit roots** — why we difference; **seasonal** vs first difference order.
2. **Invertibility / stationarity** — `statsmodels` enforcement flags.
3. **SARIMAX** — exog must be available for forecast horizon.
4. **Box–Jenkins** loop: identify (ACF/PACF) → estimate → residual check → iterate.

**Next:** Lab 5 — CatBoost on lag features.